DH演習2026 · 教材 第43版 · 2026-r43 · 2026-10-26

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第43版
教材ID: 2026-r43
教材日付: 2026-10-26
活動: source_trial_record
環境: Colab / Jupyter
ソース: a09f508a9fe4e94d0ef80ea0f2b7151c26c0ae03
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第2回：OCR結果を画像と照合し、訂正する

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：[官報 1902年8月20日](https://dl.ndl.go.jp/pid/2949042/1/5)（国立国会図書館デジタルコレクション）

## 1. 画像を取得する

`region`は切り出す範囲（左,上,幅,高さ）で、原画像のピクセルで指定します。ページ全体は`"full"`です。

In [ ]:
#@title 操作：道具を読み込む
from PIL import Image

In [ ]:
koma = 5
region = "2257,334,1059,1811"
url = f"https://dl.ndl.go.jp/api/iiif/2949042/R{koma:07d}/{region}/1520,/0/default.jpg"
!curl -fso page.jpg {url}
Image.open("page.jpg")

## 2. 文字を認識する

[NDLOCR-Lite](https://github.com/ndl-lab/ndlocr-lite/tree/d25e0d415b607ad44459ca6b95c7512a54363935)が`page.jpg`を認識し、ノートブックは結果を`text`に入れます。初回だけ、準備に約20秒かかります。

In [ ]:
ocr = "git+https://github.com/ndl-lab/ndlocr-lite@d25e0d415b607ad44459ca6b95c7512a54363935"
!uvx -q --from {ocr} ndlocr-lite --sourceimg page.jpg --output . --viz True

text = open("page.txt").read()
print(text)

## 3. 画像と照合する

枠はOCRが認識した行です。高知縣の行にあるOCR結果の「黑色椿泉」を、画像と比べてください。

In [ ]:
Image.open("viz_page.jpg")

## 4. 訂正して検索する

実行する前に、訂正後の出現回数を予測してください。

In [ ]:
query = "椿象"
corrected = text.replace("黑色椿泉", "黑色椿象")
print("訂正前:", text.count(query))
print("訂正後:", corrected.count(query))

数えているのは文字列の一致です。虫の数や報告の数ではありません。

## 5. 訂正と表記の正規化を分ける

「靜岡縣」は誤認識ではなく、資料の表記です。検索のために「静岡県」へ揃えるときは、訂正した`corrected`を残し、別の変数に入れます。

In [ ]:
query = "静岡県"
normalized = corrected.replace("靜岡縣", "静岡県")
print("正規化前:", corrected.count(query))
print("正規化後:", normalized.count(query))

## 6. 別の資料で試す

IIIFに対応した資料なら、1.の`url`をその資料の画像URLに変えると、同じ手順で認識できます。

## 確認問題

この記事で、浮塵子の発生を報告した府県はいくつですか。`text`で調べて画像と照合し、`answer`に数を入れてください。

In [ ]:
print(text.count("浮塵子"))

answer = None

In [ ]:
#@title 答えを確かめる
if answer is None:
    print("結果：不合格（`answer`に数を入れてください）")
elif answer == 7:
    print("結果：合格")
elif answer == text.count("浮塵子"):
    print("結果：不合格（文字列の一致数です。同じ府県の報告に何度も現れていないか確かめてください）")
else:
    print("結果：不合格（改行で分かれた語がないか、画像と照合してください）")